In [ ]:
# Structured Streaming - Practice Questions

This notebook tests your understanding of Spark Structured Streaming concepts including:
- Reading streaming sources (`readStream`)
- Writing streaming sinks (`writeStream`)
- Output modes (`append`, `update`, `complete`)
- Windowed aggregations
- Watermarking for late data
- Triggers
- Checkpointing
- Delta Lake as a streaming source and sink
- Change Data Feed (CDC)

**Instructions:** Each question provides context and asks you to write code in the empty cell below it. Run the setup cells first to initialize the Spark session and sample data.

## Setup: Spark Session and Sample Data

Run the cells below to initialize your Spark session and create the sample Delta tables used throughout this exercise.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col, window, current_timestamp, expr
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType
import time, shutil

spark = SparkSession.builder \
    .appName("StreamingQuestions") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session ready:", spark.version)

In [ ]:
# ── Sample Dataset: Sensor Readings ──
# This simulates IoT sensor data written to a Delta table.
# The table will serve as both a batch and streaming source for the questions below.

DELTA_PATH = "/opt/spark/delta/sensor_readings"
CHECKPOINT_BASE = "/tmp/streaming_questions_checkpoints"

# Clean up any previous runs
shutil.rmtree(CHECKPOINT_BASE, ignore_errors=True)
try:
    spark.sql(f"DROP TABLE IF EXISTS sensor_readings")
except:
    pass

sensor_schema = StructType([
    StructField("sensor_id", StringType(), False),
    StructField("temperature", DoubleType(), False),
    StructField("humidity", DoubleType(), False),
    StructField("event_time", TimestampType(), False),
    StructField("location", StringType(), False)
])

sensor_data = [
    ("sensor_01", 22.5, 45.0, "2025-06-01 10:00:00", "Building_A"),
    ("sensor_01", 23.1, 44.2, "2025-06-01 10:00:05", "Building_A"),
    ("sensor_01", 24.0, 43.8, "2025-06-01 10:00:12", "Building_A"),
    ("sensor_02", 19.8, 55.1, "2025-06-01 10:00:02", "Building_B"),
    ("sensor_02", 20.3, 54.7, "2025-06-01 10:00:08", "Building_B"),
    ("sensor_02", 20.1, 55.5, "2025-06-01 10:00:15", "Building_B"),
    ("sensor_03", 30.2, 35.0, "2025-06-01 10:00:01", "Building_A"),
    ("sensor_03", 31.5, 34.2, "2025-06-01 10:00:07", "Building_A"),
    ("sensor_03", 29.8, 36.1, "2025-06-01 10:00:20", "Building_A"),
    ("sensor_01", 25.0, 42.0, "2025-06-01 10:00:25", "Building_A"),
    ("sensor_02", 21.0, 53.0, "2025-06-01 10:00:30", "Building_B"),
    ("sensor_03", 28.5, 37.0, "2025-06-01 10:00:35", "Building_A"),
    ("sensor_01", 22.0, 46.0, "2025-06-01 10:00:40", "Building_A"),
    ("sensor_02", 19.5, 56.0, "2025-06-01 10:00:45", "Building_B"),
    ("sensor_03", 32.0, 33.0, "2025-06-01 10:00:50", "Building_A"),
]

sensor_df = spark.createDataFrame(sensor_data, schema=sensor_schema)

# Write as a CDC-enabled Delta table
sensor_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.enableChangeDataFeed", "true") \
    .save(DELTA_PATH)

print(f"Wrote {sensor_df.count()} sensor readings to Delta table at {DELTA_PATH}")
sensor_df.show(truncate=False)

---
## Question 1: Basic Streaming Read from a Rate Source

The **rate** source is a built-in streaming source that generates rows with two columns: `timestamp` and `value` (an incrementing counter). It is useful for testing streaming queries without external dependencies.

**Task:** Create a streaming DataFrame using the `rate` source that generates **2 rows per second**. Then write it to the **console** sink using `outputMode("append")` with a `processingTime` trigger of `"5 seconds"`. Let the query run for 15 seconds, then stop it.

*Hint:* Use `spark.readStream.format("rate")` and `.option("rowsPerSecond", 2)`.

In [ ]:
# Question 1: Your code here


---
## Question 2: Streaming Read from a Delta Table

Delta tables can be used as streaming sources. When you call `readStream.format("delta")`, Spark processes new rows as they are appended to the table.

**Task:** Read the sensor Delta table at `DELTA_PATH` as a stream. Write the stream to a **memory** sink with `queryName="sensor_stream"` and `outputMode("append")`. Let the query run for 10 seconds, then stop it and query the in-memory table using `spark.sql("SELECT * FROM sensor_stream")`.

*Hint:* The memory sink stores results in a temp view named by `queryName` that you can query with `spark.sql()`.

In [ ]:
# Question 2: Your code here


---
## Question 3: Output Modes — Complete vs Update vs Append

Structured Streaming supports three output modes:
- **Append**: Only new rows are written to the sink (default; only works when rows cannot change after output).
- **Update**: Only rows that changed since last trigger are written.
- **Complete**: The entire result table is written every trigger.

**Task:** Using the `rate` source (3 rows/second), compute a **running count** of rows grouped by `value % 3` (call the column `group`). Write the result to the **console** sink using **complete** mode with a trigger of `"5 seconds"`. Run for 15 seconds, then stop.

**Follow-up question (answer in a comment):** Why would `append` mode fail for this query?

In [ ]:
# Question 3: Your code here


---
## Question 4: Tumbling Window Aggregation

A **tumbling window** divides the stream into fixed, non-overlapping time intervals. Every event belongs to exactly one window.

**Task:** Read the sensor Delta table as a stream. Compute the **average temperature per sensor per 10-second tumbling window** based on `event_time`. Write the output to the **memory** sink with `queryName="q4_tumbling"` and `outputMode("complete")`. After the query processes, query the results with SQL.

*Hint:* Use `F.window(col("event_time"), "10 seconds")` and `.groupBy(window(...), col("sensor_id"))`.

In [ ]:
# Question 4: Your code here


---
## Question 5: Sliding Window Aggregation

A **sliding window** is defined by a window duration and a slide interval. Windows can overlap, so a single event may belong to multiple windows.

**Task:** Read the sensor Delta table as a stream. Compute the **max humidity per location** using a **sliding window** of `"20 seconds"` with a slide of `"10 seconds"` on `event_time`. Write to the **memory** sink with `queryName="q5_sliding"` and `outputMode("complete")`. Query the results after the stream processes.

*Hint:* Use `F.window(col("event_time"), "20 seconds", "10 seconds")`.

In [ ]:
# Question 5: Your code here


---
## Question 6: Watermarking for Late Data

In real streaming systems, events can arrive late. **Watermarking** tells Spark how long to wait for late data before finalizing a window and dropping events that arrive after the threshold.

**Task:** Read the sensor Delta table as a stream. Add a watermark of `"15 seconds"` on `event_time`. Compute the **count of readings per sensor per 10-second tumbling window**. Write results to the **memory** sink with `queryName="q6_watermark"` using **append** mode. Query the results after the stream processes.

**Follow-up question (answer in a comment):** Why does watermarking enable `append` mode for windowed aggregations? What would happen without the watermark?

In [ ]:
# Question 6: Your code here
